# SMS Spam & Phishing Detector

Classify English SMS as `ham` or `spam`, and measure how much regex fingerprint features add on top of a bag-of-words model. Specs: `docs/project-specs.md`.

## 0. Setup

Imports and the fixed random seed used everywhere (NFR-1).

In [1]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42
DATA_PATH = "data/spam.csv"

pd.set_option("display.max_colwidth", 120)

## 1. Load and deduplicate (FR-1)

Read the CSV with `latin-1` (its real encoding), keep only the label and text columns, and add a numeric target `y`.

In [2]:
raw = pd.read_csv(DATA_PATH, encoding="latin-1")
print("Raw shape:", raw.shape)
print("Columns:", list(raw.columns))

df = raw[["v1", "v2"]].rename(columns={"v1": "label", "v2": "text"})
df["y"] = (df["label"] == "spam").astype(int)
df.head()

Raw shape: (5572, 5)
Columns: ['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']


,label,text,y
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std t...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives around here though",0


A message that appeared with both labels would be contradictory training data; this check documents that the dataset has none.

In [3]:
labels_per_text = df.groupby(df["text"].str.strip())["label"].nunique()
assert (labels_per_text == 1).all(), "Some texts appear with both labels"
print("No message has conflicting labels.")

No message has conflicting labels.


Drop duplicates before any split so the same message never lands in both train and test. Duplicates are compared on trimmed text, but the `text` column itself stays raw for the regex branch.

In [4]:
n_before = len(df)
df = (df.loc[~df["text"].str.strip().duplicated(keep="first")]
        .reset_index(drop=True))

print(f"Rows before: {n_before}, after: {len(df)}, removed: {n_before - len(df)}")
print("\nClass balance after deduplication:")
print(df["label"].value_counts().to_frame("count")
        .assign(share=lambda t: (t["count"] / t["count"].sum()).round(3)))

Rows before: 5572, after: 5158, removed: 414

Class balance after deduplication:
       count  share
label              
ham     4516  0.876
spam     642  0.124


## 2. Regex fingerprint features (FR-2)

Patterns live in `features.py` so the notebook and the Streamlit app compute exactly the same features. They run on the raw `text`, before cleaning, because cleaning removes links, numbers and CAPS.

In [5]:
from features import FEATURES, FINGERPRINTS, FEATURE_NAMES, regex_features

for name, pattern in FEATURES.items():
    print(f"{name:13s} {pattern}")

n_urls        https?://\S+|www\.\S+|\b\w+\.(?:com|net|org|co\.uk)\b
n_phones      \+?\d[\d\s-]{8,}\d
n_shortcodes  \b\d{5}\b
n_money       [£$€]\s?\d+|\b\d+\s?(?:p|pounds?)\b
n_caps        \b[A-Z]{2,}\b
n_urgent      (?i)\b(?:free|win|winner|won|claim|prize|urgent|txt|reply|stop|call now)\b
n_exclaim     !


Apply the extractor to every message and add one column per feature to `df`.

In [6]:
feats = pd.DataFrame(df["text"].map(regex_features).tolist(), index=df.index)
df = pd.concat([df, feats], axis=1)
df[["label", "text"] + FEATURE_NAMES].head()

,label,text,n_urls,n_phones,n_shortcodes,n_money,n_caps,n_urgent,n_exclaim,n_chars,n_words,digit_ratio
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",0,0,0,0,0,0,0,111,20,0.00000
1,ham,Ok lar... Joking wif u oni...,0,0,0,0,0,0,0,29,6,0.00000
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std t...,0,1,1,0,2,3,0,155,28,0.16129
3,ham,U dun say so early hor... U c already then say...,0,0,0,0,0,0,0,49,11,0.00000
4,ham,"Nah I don't think he goes to usf, he lives around here though",0,0,0,0,0,0,0,61,13,0.00000


Sanity check on a typical spam message: each pattern should catch only what it is meant to.

In [7]:
example = "Call 08712300220 or text WIN to 87121"
regex_features(example)

{'n_urls': 0,
 'n_phones': 1,
 'n_shortcodes': 1,
 'n_money': 0,
 'n_caps': 1,
 'n_urgent': 1,
 'n_exclaim': 0,
 'n_chars': 37,
 'n_words': 7,
 'digit_ratio': 0.43243243243243246}

Share of messages where each fingerprint fires at least once, per class. A useful fingerprint fires often in spam and rarely in ham.

In [8]:
hit_rate = (df[FINGERPRINTS] > 0).groupby(df["label"]).mean().T.round(3)
hit_rate["spam / ham"] = (hit_rate["spam"] / hit_rate["ham"].replace(0, np.nan)).round(1)
hit_rate

label,ham,spam,spam / ham
n_urls,0.002,0.178,89.0
n_phones,0.000,0.576,NaN
n_shortcodes,0.000,0.316,NaN
n_money,0.003,0.498,166.0
n_caps,0.070,0.727,10.4
n_urgent,0.036,0.684,19.0
n_exclaim,0.117,0.469,4.0


Mean value of every feature per class, including length and digit ratio.

In [9]:
df.groupby("label")[FEATURE_NAMES].mean().T.round(3)

label,ham,spam
n_urls,0.002,0.187
n_phones,0.000,0.625
n_shortcodes,0.000,0.341
n_money,0.004,0.639
n_caps,0.381,1.983
n_urgent,0.039,1.389
n_exclaim,0.175,0.688
n_chars,70.459,137.629
n_words,14.135,23.640
digit_ratio,0.004,0.114


## 3. Train/test split

80/20, stratified on `y` so both parts keep the same spam share, with the fixed seed. The test set is used only once, for the final results table.

In [10]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2, stratify=df["y"], random_state=RANDOM_STATE)

for name, part in [("train", train), ("test", test)]:
    print(f"{name:5s} rows={len(part):5d}  spam share={part['y'].mean():.3f}")

train rows= 4126  spam share=0.125
test  rows= 1032  spam share=0.124


## 4. Model 0: regex rules, no machine learning

Rule: a message is spam if at least `k` **different** fingerprints fire. Counting distinct patterns (not total matches) stops one repeated signal, like a message in all caps, from triggering the rule on its own.

In [11]:
def n_fingerprints(features: pd.DataFrame) -> pd.Series:
    """Number of distinct regex fingerprints (0-7) that fired in each message."""
    return (features[FINGERPRINTS] > 0).sum(axis=1)


train = train.assign(n_fingerprints=n_fingerprints(train))
train.groupby("label")["n_fingerprints"].value_counts().unstack(fill_value=0)

n_fingerprints,0,1,2,3,4,5,6,7
label,,,,,,,,
ham,2908,600,100,4,0,0,0,0
spam,14,40,74,129,120,110,23,4


Tune `k` on the training set only: score every candidate and keep the one with the best spam F1.

In [12]:
from sklearn.metrics import f1_score, precision_score, recall_score

rows = []
for k in range(1, len(FINGERPRINTS) + 1):
    pred = (train["n_fingerprints"] >= k).astype(int)
    rows.append({"k": k,
                 "precision": precision_score(train["y"], pred, zero_division=0),
                 "recall": recall_score(train["y"], pred),
                 "f1": f1_score(train["y"], pred)})
k_table = pd.DataFrame(rows).set_index("k").round(3)

K_BEST = int(k_table["f1"].idxmax())
print("Best k on train:", K_BEST)
k_table

Best k on train: 3


,precision,recall,f1
k,,,
1,0.415,0.973,0.582
2,0.816,0.895,0.853
3,0.990,0.751,0.854
4,1.000,0.500,0.667
5,1.000,0.267,0.421
6,1.000,0.053,0.100
7,1.000,0.008,0.015


The final rule, reused later for cross-validation, the test set and the app.

In [13]:
def model0_predict(features: pd.DataFrame, k: int = K_BEST) -> np.ndarray:
    """Predict spam (1) when at least k distinct fingerprints fire."""
    return (n_fingerprints(features) >= k).astype(int).to_numpy()